In [3]:
import pandas as pd
df = pd.read_csv('/content/data/p03_process.csv')

In [4]:
def build_features(df): # 원본 데이터(df)를 받아 새로운 특징(변수)들을 반환하는 함수
    f = pd.DataFrame(index=df.index)
    f['temp_1']       = df['temp_1']
    f['temp_1_lag3']  = df['temp_1'].shift(3)   # 3로트 전의 상류 온도 값 매칭
    f['temp_1_ma5']   = df['temp_1'].rolling(5).mean()  # 5개 로트 온도의 이동평균(이때 앞 4개는 NaN)
    f['press_1']      = df['press_1']
    f['time_1']       = df['time_1']
    f['time_dev2']    = (df['time_1'] - 60) ** 2 # 60초에서 멀어질수록 값이 기하급수적으로 커짐(이탈을 쉽게 파악)
    f['temp_2']       = df['temp_2']
    f['amb_temp']     = df['amb_temp']
    f['amb_humid']    = df['amb_humid']
    # 3로트 전 온도 기준(180)과 현재 습도가 기준(50)이 벗어난 정도를 곱한다.
    # 두 조건이 동시에 크게 어긋날 때 치수에 미치는 복합적인 영향을 모델에 알려준다.
    f['temp_x_humid'] = (f['temp_1_lag3'] - 180) * (df['amb_humid'] - 50)
    # 3로트전 온도 매칭, 이동평균추가, 특정 치수 틀어짐을 강력하게 표현
    return f


In [ ]:
X = build_features(df)
y = df['dim']
# 모든 컬럼(axis=1)에 빈칸이 하나도 없는 행만 True로 표시하는 필터(mask) 생성
mask = X.notna().all(axis=1)
# 결측치 없는 값만 각각 넣어준다.
X, y = X[mask], y[mask]

In [7]:
X.to_csv('/content/data/X_features.csv', index=False)
y.to_csv('/content/data/y_target.csv', index=False)